In [ ]:
# mobilenet_pest.py
import torch, time, os
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
import torch.nn as nn
import numpy as np

train_dir = '/path/to/ip102/train'
val_dir   = '/path/to/ip102/val'
num_classes = 102  # for full IP102; adjust if subset

# transforms
tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
train_ds = datasets.ImageFolder(train_dir, transform=tf)
val_ds   = datasets.ImageFolder(val_dir, transform=tf)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=8)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=8)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)  # replace final FC
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# training loop similar to EfficientNet example (omitted here to conserve space)
# after training, evaluate metrics:

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc

# Measure model size (in-memory param count)
param_count = sum(p.numel() for p in model.parameters())
print(f"Parameters: {param_count:,}")

# Measure inference time (single image avg)
model.eval()
dummy = torch.randn(1,3,224,224).to(device)
# warm-up
for _ in range(10):
    _ = model(dummy)
N=100
t0 = time.time()
for _ in range(N):
    _ = model(dummy)
t = (time.time()-t0)/N
print(f"Avg inference time per image: {t*1000:.2f} ms")